In [ ]:
from syncode import Syncode
from pathlib import Path
import subprocess
import tempfile

In [ ]:
model_name = "microsoft/phi-2"

# Load the Syncode augmented model
syn_llm = Syncode(
    model=model_name,
    mode='grammar_mask',
    parse_output_only=False,
    grammar="c",
)

In [ ]:
def test_examples(path: Path, verbose: bool = False):
    results = {}
    for source_path in sorted(path.glob("*.c")):
        # Read the source code stub from the file and generate a completion using the Syncode model
        example_stub = source_path.read_text(encoding="utf-8")
        completion = syn_llm.infer(example_stub, stop_words=["\n}"])[0] # type: ignore
        completed_program = example_stub + completion

        # Write the completed program to a temporary file and run GCC syntax check
        with tempfile.TemporaryDirectory() as temp_dir:
            generated_path = Path(temp_dir) / source_path.name
            generated_path.write_text(completed_program, encoding="utf-8")
            result = subprocess.run(
                ["gcc", "-c", "-fsyntax-only", "-Wno-implicit-function-declaration", str(generated_path)],
                capture_output=True,
                text=True,
                check=False,
            )

        # Store the results for this source file
        if_passed = (result.returncode == 0)
        results[source_path.name] = {
            "completed_program": completed_program,
            "gcc_result": result,
            "passed": if_passed,
        }
    
        if verbose:
            print(f"--- {source_path.name} ---")
            print(completed_program)
            if result.stdout:
                print(result.stdout)
            if result.stderr:
                print(result.stderr)
            print(f"GCC syntax check: {if_passed}")
            print()

    return results


In [ ]:
test_examples(Path("./c_stubs/typedef_advanced"), verbose=True)